# Transformação Silver

Limpeza, tipagem e padronização das chaves de cruzamento. É aqui que entram as regras de
negócio.

Duas decisões estruturais desta camada:

1. **Recorte da rede pública.** O IDEB traz linhas por rede (`Estadual`, `Municipal`,
   `Federal`, `Pública`); mantém-se apenas `Pública`, que é a consolidação já feita pelo
   próprio INEP e a rede de interesse do problema de negócio. No Censo, o equivalente é
   `TP_DEPENDENCIA` de 1 a 3 (Federal, Estadual, Municipal).
2. **Mudança de grão do Censo.** Os microdados vêm por escola; aqui viram indicadores
   municipais. Como as colunas `IN_*` são binárias (0/1, conforme o dicionário oficial do
   INEP), a média delas × 100 é o percentual de escolas do município que têm o recurso.

In [ ]:
from pyspark.sql import functions as F

import os
import sys

# Funções compartilhadas entre os notebooks ficam em pipeline_utils.py, na mesma pasta.
_pasta = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
sys.path.insert(0, _pasta if _pasta.startswith("/Workspace") else f"/Workspace{_pasta}")
from pipeline_utils import criar_tabela, garantir_chave_unica, tags_silver  # noqa: E402

# Parâmetro: usado pelo Job (Jobs & Pipelines) e com padrão para execução interativa.
dbutils.widgets.text("catalog", "workspace")

CATALOG = dbutils.widgets.get("catalog")

spark.sql(f"USE CATALOG {CATALOG}")


## Municípios - dimensão enriquecida com população

A população do Censo 2022 é o denominador de toda métrica per capita da camada Gold. Sem
ela, comparar valores absolutos entre municípios apenas reproduz o tamanho da população:
na validação prévia, a correlação do Bolsa Família com o IDEB foi de -0,041 em valores
absolutos contra -0,490 per capita.

In [ ]:
df_municipios = (
    spark.table(f"{CATALOG}.bronze.municipios_ibge")
    .join(
        spark.table(f"{CATALOG}.bronze.populacao_municipios")
             .select("codigo_municipio_ibge", "populacao"),
        "codigo_municipio_ibge",
        "left",
    )
    .select("codigo_municipio_ibge", "nome_municipio", "sigla_uf", "nome_regiao", "populacao")
)

garantir_chave_unica(df_municipios, ["codigo_municipio_ibge"], "municipios")

criar_tabela(
    df_municipios,
    f"{CATALOG}.silver.municipios",
    "Municípios brasileiros com UF, região e população do Censo 2022. Base da dimensão de município.",
    propriedades={"camada": "silver", "grao": "municipio", "origem": "bronze.municipios_ibge + bronze.populacao_municipios"},
    tags=tags_silver("ibge", "territorio"),
    comentarios={
        "codigo_municipio_ibge": "Código IBGE do município, 7 dígitos. Chave única da tabela.",
        "nome_municipio": "Nome oficial do município.",
        "sigla_uf": "Sigla da unidade da federação.",
        "nome_regiao": "Região geográfica. Domínio: Norte, Nordeste, Centro-Oeste, Sudeste, Sul.",
        "populacao": "População residente no Censo 2022. Nula para municípios instalados depois do Censo.",
    },
)

print(f"silver.municipios: {df_municipios.count():,} linhas")

## IDEB - rede pública, tipado e com etapa de ensino

`VL_OBSERVADO_2023` é a nota do IDEB da edição 2023 (o nome não contém "IDEB"). A etapa de
ensino não existe como coluna na planilha do INEP: foi materializada na extração, a partir
do arquivo de origem.

In [ ]:
df_ideb = (
    spark.table(f"{CATALOG}.bronze.ideb_municipios")
    .where(F.col("REDE") == "Pública")
    .select(
        F.col("CO_MUNICIPIO").cast("long").alias("codigo_municipio_ibge"),
        F.lit(2023).alias("ano"),
        F.col("etapa_ensino"),
        F.col("VL_OBSERVADO_2023").cast("double").alias("vl_ideb"),
        F.col("VL_NOTA_MATEMATICA_2023").cast("double").alias("vl_nota_matematica"),
        F.col("VL_NOTA_PORTUGUES_2023").cast("double").alias("vl_nota_portugues"),
        F.col("VL_INDICADOR_REND_2023").cast("double").alias("vl_indicador_rendimento"),
    )
)
garantir_chave_unica(df_ideb, ["codigo_municipio_ibge", "ano", "etapa_ensino"], "ideb")

criar_tabela(
    df_ideb,
    f"{CATALOG}.silver.ideb",
    "IDEB 2023 e proficiências do SAEB da rede pública, por município e etapa de ensino.",
    propriedades={"camada": "silver", "grao": "municipio_ano_etapa", "origem": "bronze.ideb_municipios",
                  "filtro": "REDE = Pública"},
    tags=tags_silver("inep", "educacao"),
    comentarios={
        "codigo_municipio_ibge": "Código IBGE do município, 7 dígitos.",
        "ano": "Ano da edição do IDEB.",
        "etapa_ensino": "Domínio: Anos Iniciais, Anos Finais.",
        "vl_ideb": "IDEB observado (VL_OBSERVADO_2023). Escala de 0 a 10.",
        "vl_nota_matematica": "Proficiência média em Matemática no SAEB. Escala SAEB.",
        "vl_nota_portugues": "Proficiência média em Língua Portuguesa no SAEB. Escala SAEB.",
        "vl_indicador_rendimento": "Indicador de rendimento (fluxo escolar). Escala de 0 a 1.",
    },
)

print(f"silver.ideb: {df_ideb.count():,} linhas")
display(df_ideb.groupBy("etapa_ensino").agg(
    F.count("*").alias("municipios"),
    F.round(F.avg("vl_ideb"), 2).alias("ideb_medio"),
    F.sum(F.when(F.col("vl_ideb").isNull(), 1).otherwise(0)).alias("sem_nota"),
))

## Censo Escolar - de escola para município

Filtra escolas em atividade da rede pública e converte cada flag de infraestrutura no
percentual de escolas do município que a possuem.

In [ ]:
INDICADORES_INFRA = [
    "IN_INTERNET", "IN_INTERNET_ALUNOS", "IN_COMPUTADOR", "IN_LABORATORIO_INFORMATICA",
    "IN_BIBLIOTECA", "IN_LABORATORIO_CIENCIAS", "IN_QUADRA_ESPORTES", "IN_REFEITORIO",
    "IN_AGUA_POTAVEL", "IN_ESGOTO_REDE_PUBLICA", "IN_ENERGIA_REDE_PUBLICA",
    "IN_LIXO_SERVICO_COLETA", "IN_BANHEIRO", "IN_ALIMENTACAO",
    "IN_ACESSIBILIDADE_RAMPAS", "IN_BANHEIRO_PNE",
]

agregacoes = [
    F.count("*").alias("qt_escolas_publicas"),
    F.sum(F.col("QT_MAT_BAS").cast("double")).alias("qt_matriculas"),
    F.sum(F.col("QT_DOC_BAS").cast("double")).alias("qt_docentes"),
] + [
    F.round(F.avg(F.col(ind).cast("double")) * 100, 2).alias(f"pct_escolas_{ind.replace('IN_', '').lower()}")
    for ind in INDICADORES_INFRA
]

df_censo = (
    spark.table(f"{CATALOG}.bronze.censo_escolar_escolas")
    .where((F.col("TP_SITUACAO_FUNCIONAMENTO") == 1) & (F.col("TP_DEPENDENCIA").isin(1, 2, 3)))
    .groupBy(
        F.col("CO_MUNICIPIO").cast("long").alias("codigo_municipio_ibge"),
        F.col("NU_ANO_CENSO").cast("int").alias("ano"),
    )
    .agg(*agregacoes)
)

# Descrição de cada percentual vem do dicionário oficial do INEP.
dicionario = {
    l["nome_variavel"]: l["descricao"]
    for l in spark.table(f"{CATALOG}.bronze.dicionario_censo_escolar").collect()
}
comentarios_censo = {
    "codigo_municipio_ibge": "Código IBGE do município, 7 dígitos.",
    "ano": "Ano do Censo Escolar.",
    "qt_escolas_publicas": "Escolas públicas (federal, estadual e municipal) em atividade no município.",
    "qt_matriculas": "Matrículas da educação básica nessas escolas (soma de QT_MAT_BAS).",
    "qt_docentes": "Docentes da educação básica nessas escolas (soma de QT_DOC_BAS).",
}
for indicador in INDICADORES_INFRA:
    coluna = f"pct_escolas_{indicador.replace('IN_', '').lower()}"
    comentarios_censo[coluna] = (
        f"Percentual de escolas públicas em atividade com: {dicionario.get(indicador, indicador)}. "
        f"Derivado de {indicador}. Escala de 0 a 100."
    )

criar_tabela(
    df_censo,
    f"{CATALOG}.silver.censo_escolar_infra",
    "Indicadores de infraestrutura das escolas públicas em atividade, agregados por município.",
    propriedades={"camada": "silver", "grao": "municipio_ano", "origem": "bronze.censo_escolar_escolas",
                  "filtro": "TP_SITUACAO_FUNCIONAMENTO = 1 e TP_DEPENDENCIA em 1, 2, 3"},
    tags=tags_silver("inep", "educacao"),
    comentarios=comentarios_censo,
)

print(f"silver.censo_escolar_infra: {df_censo.count():,} municípios")

## Bolsa Família

Já chega agregado por município (a agregação aconteceu na extração, porque o arquivo bruto
traz nome e NIS de cada beneficiário e o edital veda dados pessoais no lakehouse).

In [ ]:
df_bolsa_familia = (
    spark.table(f"{CATALOG}.bronze.bolsa_familia_municipio")
    .select(
        F.col("codigo_municipio_ibge").cast("long").alias("codigo_municipio_ibge"),
        F.substring(F.col("mes_referencia"), 1, 4).cast("int").alias("ano"),
        F.col("mes_referencia"),
        F.col("valor_total").cast("double"),
        F.col("quantidade_beneficiados").cast("long"),
    )
)
garantir_chave_unica(df_bolsa_familia, ["codigo_municipio_ibge", "mes_referencia"], "bolsa_familia")

criar_tabela(
    df_bolsa_familia,
    f"{CATALOG}.silver.bolsa_familia",
    "Novo Bolsa Família por município no mês de referência, sem dados pessoais.",
    propriedades={"camada": "silver", "grao": "municipio_mes", "origem": "bronze.bolsa_familia_municipio"},
    tags=tags_silver("portal_transparencia", "assistencia_social"),
    comentarios={
        "codigo_municipio_ibge": "Código IBGE do município, 7 dígitos.",
        "ano": "Ano do mês de referência.",
        "mes_referencia": "Mês de competência dos pagamentos (AAAAMM).",
        "valor_total": "Soma das parcelas pagas no município, em R$.",
        "quantidade_beneficiados": "Quantidade de benefícios pagos no município.",
    },
)

print(f"silver.bolsa_familia: {df_bolsa_familia.count():,} linhas")

## Validação da camada

Nenhuma chave nula e nenhum código de município fora da base oficial do IBGE.

In [0]:
codigos_ibge = {
    linha["codigo_municipio_ibge"]
    for linha in spark.table(f"{CATALOG}.silver.municipios")
                      .select("codigo_municipio_ibge").collect()
}

for tabela in ["municipios", "ideb", "censo_escolar_infra", "bolsa_familia"]:
    df = spark.table(f"{CATALOG}.silver.{tabela}")
    nulos = df.where(F.col("codigo_municipio_ibge").isNull()).count()
    codigos = {l["codigo_municipio_ibge"] for l in df.select("codigo_municipio_ibge").distinct().collect()}
    orfaos = codigos - codigos_ibge
    assert nulos == 0, f"{tabela}: {nulos} chaves nulas"
    assert not orfaos, f"{tabela}: {len(orfaos)} códigos fora da base do IBGE"
    print(f"{tabela:22} {df.count():>8,} linhas | {len(codigos):>5,} municípios | 0 nulos | 0 órfãos")